In [1]:
import boto3
import sagemaker
from time import strftime, gmtime
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

default_bucket = sess.default_bucket()
default_bucket_prefix = sess.default_bucket_prefix

print(default_bucket)
print(role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker-us-east-1-635026339135
arn:aws:iam::635026339135:role/SageMakerStudioExecutionRole2026


In [2]:
processing_image = "635026339135.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-processing:latest"
print(processing_image)

635026339135.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-processing:latest


In [4]:
RAW_DIR = "/home/sagemaker-user/Emi-y-Mora/data/raw"

prefix = f"emi-y-mora-raw-{strftime('%Y%m%d-%H%M%S', gmtime())}"
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

raw_data_s3 = sess.upload_data(
    path=RAW_DIR,
    bucket=default_bucket,
    key_prefix=prefix,
)

print(raw_data_s3)

s3://sagemaker-us-east-1-635026339135/emi-y-mora-raw-20260313-171651


In [5]:
output_prefix = f"emi-y-mora-processing-output-{strftime('%Y%m%d-%H%M%S', gmtime())}"
if default_bucket_prefix:
    output_prefix = f"{default_bucket_prefix}/{output_prefix}"

output_s3 = f"s3://{default_bucket}/{output_prefix}"
print(output_s3)

s3://sagemaker-us-east-1-635026339135/emi-y-mora-processing-output-20260313-171655


In [6]:
processor = ScriptProcessor(
    image_uri=processing_image,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=sess,
)

In [7]:
processor.run(
    code="/home/sagemaker-user/Emi-y-Mora/processing/preprocess.py",
    inputs=[
        ProcessingInput(
            source=raw_data_s3,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=output_s3,
        )
    ],
    arguments=["--lags", "1,2,3"],
    logs=True,
)

INFO:sagemaker:Creating processing-job with name emi-y-mora-processing-2026-03-13-17-17-45-949


...............
..

In [8]:
latest_job = processor.jobs[-1]
print(latest_job.describe()["ProcessingJobStatus"])
print(latest_job.describe()["ProcessingJobName"])

Completed
emi-y-mora-processing-2026-03-13-17-17-45-949


In [9]:
import boto3

s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=default_bucket, Prefix=output_prefix)

for obj in resp.get("Contents", []):
    print(obj["Key"])

emi-y-mora-processing-output-20260313-171655/X_train.csv
emi-y-mora-processing-output-20260313-171655/X_valid.csv
emi-y-mora-processing-output-20260313-171655/y_train.csv
emi-y-mora-processing-output-20260313-171655/y_valid.csv


In [10]:
import pandas as pd

x_train_s3 = f"{output_s3}/X_train.csv"
df = pd.read_csv(x_train_s3)
df.head()

,date_block_num,shop_id,item_id,item_category_id,month,year,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3
0,1,0,12,55,1,0,0.0,0.0,0.0
1,0,0,19,40,0,0,0.0,0.0,0.0
2,0,0,27,19,0,0,0.0,0.0,0.0
3,1,0,27,19,1,0,0.0,0.0,0.0
4,0,0,28,30,0,0,0.0,0.0,0.0
